# 3. Wealth-related inequality and sensitivity analyses

This notebook is step 3 of 5. It needs the derived data from notebook 1.

| Section | What it does | Manuscript |
|---|---|---|
| 1 | Concentration indices with bootstrap CIs, concentration curves, decomposition | Results 3.4, Table SA16, Fig. 2 |
| 2 | Sensitivity analyses S0–S12 on both scales | Results 3.5, Table SA8 |

**Concentration indices**

| Index | Use |
|---|---|
| Erreygers | Main measure: corrected for a binary outcome |
| Wagstaff | Normalised index, for comparison |
| Relative | Standard index, for comparison |

A positive index means GDM is concentrated among wealthier women. Women are ranked by the continuous DHS wealth score.

## Setup

The Rao–Wu bootstrap weights are computed once and used in both sections: 1,000 replicates, with primary sampling units resampled within strata.

In [1]:
# Find the project folder, wherever the notebook was opened from
root <- normalizePath(getwd(), winslash = "/")
while (!file.exists(file.path(root, "gdm-project.Rproj"))) {
  if (dirname(root) == root) stop("Open this notebook inside the gdm-project folder.")
  root <- dirname(root)
}
setwd(root)

# Use the package versions locked in renv.lock
if (!nzchar(Sys.getenv("RENV_PROJECT"))) {
  Sys.setenv(RENV_CONFIG_NAMESPACES_CHECK = "FALSE")
  source("renv/activate.R")
}
options(width = 120)

# Settings (config/config.yml) and analysis functions (src/)
source(here::here("src", "common", "setup.R"))

# Data saved by notebook 1
women  <- readRDS(data_file("women"))
sample <- readRDS(data_file("analytic_sample"))
des  <- svy_domain(sample, women)

# Rao-Wu rescaled bootstrap weights (one column per replicate)
boot <- bootstrap_weights(sample, women, cfg)

## 1. Inequality on both glucose scales

For each scale, this loop computes:

- the three indices and their bootstrap CIs
- the concentration curve
- the decomposition of the Erreygers index, without wealth (main) and with wealth (supplementary)

Wealth is left out of the main decomposition because it is the ranking variable.

Functions: `src/common/inequality.R`.

In [2]:
indices <- list(); curve <- list(); ci <- list(); dec <- list(); dec_w <- list()
for (s in names(scales)) {
  o <- scales[[s]]
  keep <- !is.na(sample[[o]])
  d <- sample[keep, ]

  # Point estimates and the concentration curve
  est <- inequality_indices(d[[o]], d$wealth_score, d$weight)
  indices[[s]] <- cbind(scale = s, est)
  curve[[s]] <- cbind(scale = s, concentration_curve(d[[o]], d$wealth_score, d$weight))

  # Bootstrap CIs for the three indices
  reps <- bootstrap_indices(d[[o]], d$wealth_score, boot$weights[keep, ])
  ci[[s]] <- cbind(scale = s, index = c("relative", "wagstaff", "erreygers"), rbind(
    bootstrap_summary(est$C_relative, reps[1, ], boot),
    bootstrap_summary(est$C_wagstaff, reps[2, ], boot),
    bootstrap_summary(est$C_erreygers, reps[3, ], boot)))

  # Decomposition of the Erreygers index, with bootstrap CIs
  dec[[s]] <- cbind(scale = s, decomposition_with_ci(sample, cfg$decomp_covariates, boot, o))
  dec_w[[s]] <- cbind(scale = s, decomposition_with_ci(sample, c(cfg$decomp_covariates, "wealth"), boot, o))
}
ci <- do.call(rbind, ci)

save_table(do.call(rbind, indices), "inequality_indices")
save_table(do.call(rbind, curve), "concentration_curve")
save_table(ci, "inequality_bootstrap")
save_table(ci[ci$index == "erreygers", ], "erreygers_bootstrap")
save_table(do.call(rbind, dec), "decomposition_main")
save_table(do.call(rbind, dec_w), "decomposition_with_wealth")

### Key results

In [3]:
# Indices with their bootstrap 95% CIs
ci_view <- ci[c("scale", "index", "estimate", "ci_lower", "ci_upper")]
ci_view[3:5] <- round(ci_view[3:5], 3)
ci_view

                    scale     index estimate ci_lower ci_upper
released.1       released  relative    0.092    0.033    0.157
released.2       released  wagstaff    0.250    0.093    0.415
released.3       released erreygers    0.233    0.084    0.386
unconverted.1 unconverted  relative    0.109   -0.006    0.233
unconverted.2 unconverted  wagstaff    0.162   -0.009    0.339
unconverted.3 unconverted erreygers    0.141   -0.008    0.298


In [4]:
# Contribution of each covariate to the Erreygers index
decomposition <- do.call(rbind, dec)
dec_view <- decomposition[c("scale", "term", "contribution", "contribution_ci_lower", "contribution_ci_upper", "percent")]
dec_view[3:6] <- round(dec_view[3:6], 3)
dec_view

                    scale               term contribution contribution_ci_lower contribution_ci_upper percent
released.1       released       age_group25+        0.011                -0.005                 0.044   4.614
released.2       released     residenceUrban        0.048                 0.006                 0.104  20.517
released.3       released educationSecondary       -0.001                -0.018                 0.015  -0.392
released.4       released    educationHigher        0.027                -0.051                 0.096  11.633
released.5       released       trimester2nd        0.026                 0.000                 0.069  11.156
released.6       released       trimester3rd       -0.015                -0.055                 0.021  -6.640
released.7       released                bmi        0.004                -0.030                 0.043   1.609
released.8       released           residual        0.134                 0.020                 0.248  57.502
unconverte

## 2. Sensitivity analyses

| ID | Change from the main analysis |
|---|---|
| S0 | Main analysis |
| S1 | Outcome includes the diabetes range (FPG 5.1 mmol/L or above) |
| S2 | Restricted to 6 or more completed months of gestation |
| S3 | Excluding completed months 1–2 |
| S4 | Age cut-off at 30 years |
| S5 | Continuous age instead of age group |
| S6 | Additional adjustment for education |
| S7 | Additional adjustment for hypertension |
| S8 | Measured BMI instead of estimated pre-pregnancy BMI |
| S9 | Excluding women not tested on a fasted-appointment day |
| S10 | Unweighted prevalence with an exact binomial CI |
| S11 | Excluding women who drank other than water within 8 hours |
| S12 | National wealth quintiles in three groups instead of tertiles |

Each scenario lists its outcome, covariates and rows. `ci = TRUE` means the sample changes, so the Erreygers index is recomputed.

In [5]:
x <- cfg$covariates
all_rows <- rep(TRUE, nrow(sample))

scenarios <- list(
  S0 = list(label = "Main analysis", outcome = "gdm", covariates = x, rows = all_rows, ci = TRUE),
  S1 = list(label = "Outcome includes FPG 7.0 mmol/L or above", outcome = "gdm_incl_dm",
            covariates = x, rows = all_rows, ci = TRUE),
  S2 = list(label = "Restricted to 6 or more months of gestation", outcome = "gdm",
            covariates = setdiff(x, "trimester"), rows = sample$v214 >= 6, ci = TRUE),
  S3 = list(label = "Excluding months 1-2 of gestation", outcome = "gdm",
            covariates = x, rows = sample$v214 >= 3, ci = TRUE),
  S4 = list(label = "Age cut-off at 30 years", outcome = "gdm",
            covariates = sub("age_group", "age_group_alt", x), rows = all_rows, ci = FALSE),
  S5 = list(label = "Continuous age", outcome = "gdm",
            covariates = sub("age_group", "age_c", x), rows = all_rows, ci = FALSE),
  S6 = list(label = "Additionally adjusted for education", outcome = "gdm",
            covariates = c(x, "education"), rows = all_rows, ci = FALSE),
  S7 = list(label = "Additionally adjusted for hypertension", outcome = "gdm",
            covariates = c(x, "hypertension"), rows = all_rows, ci = FALSE),
  S8 = list(label = "Measured BMI instead of estimated pre-pregnancy BMI", outcome = "gdm",
            covariates = sub("^bmi$", "bmi_measured", x), rows = all_rows, ci = FALSE),
  S9 = list(label = "Excluding women not tested on a fasted-appointment day", outcome = "gdm",
            covariates = x, rows = sample$tested_on_fasted_day, ci = TRUE),
  S11 = list(label = "Excluding women who drank other than water within 8 hours", outcome = "gdm",
             covariates = x, rows = !sample$drank_8h, ci = TRUE),
  S12 = list(label = "National wealth quintiles (poorest two, middle, richest two)", outcome = "gdm",
             covariates = sub("^wealth$", "wealth_v190", x), rows = all_rows, ci = FALSE))

# Outcome variable of a scenario on a given scale
on_scale <- function(outcome, s) if (s == "unconverted") paste0(outcome, "_alt") else outcome

### Run every scenario on both scales

In [6]:
prev <- list(); eff <- list(); ineq <- list()
for (s in names(scales)) for (id in names(scenarios)) {
  sc <- scenarios[[id]]
  o <- on_scale(sc$outcome, s)
  key <- paste(s, id)
  d_s <- subset(des, sc$rows)

  # Prevalence and adjusted prevalence ratios
  prev[[key]] <- cbind(scale = s, id = id, scenario = sc$label, weighted_prevalence(d_s, o))
  eff[[key]] <- cbind(scale = s, id = id, scenario = sc$label,
                      effect_table(fit_pr(d_s, sc$covariates, o), "PR", "adjusted"))

  # Erreygers index, when the scenario changes the sample or the outcome
  if (sc$ci) {
    keep <- sc$rows & !is.na(sample[[o]])
    d <- sample[keep, ]
    est <- 4 * weighted.mean(d[[o]], d$weight) * concentration_index(d[[o]], d$wealth_score, d$weight)
    reps <- bootstrap_erreygers(d[[o]], d$wealth_score, boot$weights[keep, ])
    ineq[[key]] <- cbind(scale = s, id = id, scenario = sc$label, bootstrap_summary(est, reps, boot))
  }
}

# S10: unweighted prevalence with an exact binomial CI
for (s in names(scales)) {
  y <- sample[[scales[[s]]]]
  y <- y[!is.na(y)]
  bt <- binom.test(sum(y), length(y))
  prev[[paste(s, "S10")]] <- data.frame(scale = s, id = "S10", scenario = "Unweighted (exact binomial CI)",
                                        n = length(y), events = sum(y), prevalence = mean(y), se = NA,
                                        ci_lower = bt$conf.int[1], ci_upper = bt$conf.int[2])
}

save_table(do.call(rbind, prev), "sensitivity_prevalence")
eff <- do.call(rbind, eff)
eff$term <- sub("^wealth_v190", "wealth", eff$term)   # S12: same rows of the table as the tertiles
save_table(eff, "sensitivity_effects")
save_table(do.call(rbind, ineq), "sensitivity_inequality")

### Key results

The adjusted PR for the highest wealth tertile in each scenario.

In [7]:
wealth_view <- eff[eff$term == "wealthHighest", c("scale", "id", "n", "estimate", "ci_lower", "ci_upper")]
wealth_view[4:6] <- round(wealth_view[4:6], 2)
rownames(wealth_view) <- NULL
wealth_view

         scale  id   n estimate ci_lower ci_upper
1     released  S0 264     1.38     1.02     1.86
2     released  S1 270     1.35     1.01     1.79
3     released  S2 120     1.28     0.76     2.13
4     released  S3 223     1.35     0.94     1.94
5     released  S4 264     1.40     1.04     1.89
6     released  S5 264     1.38     1.03     1.87
7     released  S6 264     1.39     1.01     1.92
8     released  S7 264     1.38     1.03     1.86
9     released  S8 264     1.38     1.03     1.86
10    released  S9 261     1.40     1.04     1.89
11    released S11 259     1.40     1.04     1.89
12    released S12 264     1.40     1.07     1.82
13 unconverted  S0 267     1.17     0.70     1.95
14 unconverted  S1 270     1.14     0.69     1.90
15 unconverted  S2 123     0.83     0.36     1.89
16 unconverted  S3 226     1.14     0.57     2.30
17 unconverted  S4 267     1.17     0.70     1.94
18 unconverted  S5 267     1.15     0.69     1.92
19 unconverted  S6 267     1.32     0.76     2.28


**Next:** `04_validity`.